# 2.1 Landing

Creates unmaterialized views directly over the raw files in the managed Volume. No data is copied — each view reads from the files at query time.

**Why unmaterialized views?**
- No extra storage cost
- Always reflect the latest files dropped into the Volume
- Defer deduplication and type-casting to downstream layers

**Datasets:**

| View | Format | Source path |
|------|--------|-------------|
| `v_customers` | JSON | `/customers/` |
| `v_addresses` | Tab-delimited CSV | `/addresses/` |
| `v_orders` | Raw text (malformed JSON) | `/orders/` |

In [0]:
%fs ls /Volumes/de_lab/landing/de_lab_vol/

## Customers

In [0]:
SELECT *
FROM json.`/Volumes/de_lab/landing/de_lab_vol/customers/`
LIMIT 10;

In [0]:
CREATE OR REPLACE VIEW de_lab.landing.v_customers AS
SELECT *
FROM json.`/Volumes/de_lab/landing/de_lab_vol/customers/`;

In [0]:
SELECT * FROM de_lab.landing.v_customers LIMIT 10;

## Addresses

Addresses are tab-delimited CSV. `read_files` lets us specify the format and delimiter explicitly.

In [0]:
SELECT *
FROM read_files(
  '/Volumes/de_lab/landing/de_lab_vol/addresses/',
  format    => 'csv',
  delimiter => '\t',
  header    => true
)
LIMIT 5;

In [0]:
CREATE OR REPLACE VIEW de_lab.landing.v_addresses AS
SELECT *
FROM read_files(
  '/Volumes/de_lab/landing/de_lab_vol/addresses/',
  format    => 'csv',
  delimiter => '\t',
  header    => true
);

In [0]:
SELECT * FROM de_lab.landing.v_addresses LIMIT 10;

## Orders — Raw Text

The order files contain malformed JSON: date values are unquoted (`"order_date": 2024-01-01` instead of `"order_date": "2024-01-01"`). Spark's JSON reader silently returns NULLs for those fields. We land the raw text and repair the JSON in the Silver layer.

In [0]:
-- Attempting the JSON reader — order_date parses as NULL due to malformed input
SELECT *
FROM json.`/Volumes/de_lab/landing/de_lab_vol/orders/`
LIMIT 5;

In [0]:
-- Reading as plain text preserves each order as a raw string we can fix downstream
SELECT *
FROM text.`/Volumes/de_lab/landing/de_lab_vol/orders/`
LIMIT 5;

In [0]:
CREATE OR REPLACE VIEW de_lab.landing.v_orders AS
SELECT *
FROM text.`/Volumes/de_lab/landing/de_lab_vol/orders/`;

In [0]:
SELECT * FROM de_lab.landing.v_orders LIMIT 5;